# MiniGenie — Flow Matching Dynamics Training

Train the flow matching U-Net dynamics model on CoinRun episodes.  
**Spec:** `docs/build_spec.md` §2.3–2.4 — U-Net with AdaGN, flow matching ODE, CFG.  
**Config:** `configs/dynamics.yaml` — 150K steps, batch 16, lr 2e-4→1e-5, fp16, noise aug.

### Prerequisites
- VQ-VAE must be trained first (for future latent-space training — currently training in pixel space)
- CoinRun data extracted on Drive

### Targets
- Single-step PSNR > 22 dB
- Flow matching loss in [0.01, 0.05] range

### Expected runtime
- T4: ~18–24 hours for 150K steps (multiple sessions)
- A100: ~6–10 hours for 150K steps

---
## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/minigenie'
os.makedirs(f'{DRIVE_PROJECT}/checkpoints/dynamics', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/samples_dynamics', exist_ok=True)
print(f'Drive project root: {DRIVE_PROJECT}')

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/minigenie.git'  # <-- UPDATE THIS
LOCAL_CODE = '/content/minigenie'

if os.path.exists(LOCAL_CODE):
    !cd {LOCAL_CODE} && git pull --ff-only
else:
    !git clone {REPO_URL} {LOCAL_CODE}

os.chdir(LOCAL_CODE)
!git log --oneline -3

In [ ]:
!pip install -q einops pyyaml wandb tqdm imageio scipy pillow matplotlib torchvision
!pip install -q -e .

import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > GPU'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

# Recommend batch size based on GPU
if gpu_mem >= 35:
    print('A100 detected -- default batch_size=16 should be fine.')
elif gpu_mem >= 14:
    print('T4 detected -- may need batch_size=8 or gradient_accumulation_steps=2.')
else:
    print(f'Small GPU ({gpu_mem:.0f} GB). Use batch_size=4 + gradient accumulation.')

---
## 2. Extract & Verify Data

In [ ]:
import glob
import numpy as np

DATA_TAR = f'{DRIVE_PROJECT}/coinrun_data.tar.gz'
LOCAL_DATA = '/content/minigenie/data/coinrun/episodes'

if os.path.exists(LOCAL_DATA) and len(os.listdir(LOCAL_DATA)) > 100:
    print('Data already extracted.')
elif os.path.exists(DATA_TAR):
    print('Extracting data...')
    !tar xzf {DATA_TAR} -C /content/minigenie/
else:
    raise FileNotFoundError(
        f'Data archive not found at {DATA_TAR}\n'
        'Upload coinrun_data.tar.gz to Google Drive at MyDrive/minigenie/'
    )

paths = sorted([p for p in glob.glob(f'{LOCAL_DATA}/*.npz')
                if not os.path.basename(p).startswith('._')])
print(f'Episodes: {len(paths)}')

# Spot check -- verify sequence structure for dynamics training
ep = np.load(paths[0])
frames, actions = ep['frames'], ep['actions']
print(f'Sample episode: {frames.shape[0]} frames, {actions.shape[0]} actions')
print(f'  frames: {frames.shape} {frames.dtype}')
print(f'  actions: {actions.shape} range [{actions.min()}, {actions.max()}]')
assert frames.shape[0] == actions.shape[0] + 1, 'Frame/action count mismatch'
assert len(paths) >= 1000
print('Data OK')

---
## 3. Symlink Checkpoints & Samples to Drive

In [ ]:
import shutil

symlinks = {
    '/content/minigenie/checkpoints/dynamics': f'{DRIVE_PROJECT}/checkpoints/dynamics',
    '/content/minigenie/samples_dynamics': f'{DRIVE_PROJECT}/samples_dynamics',
}

for local, remote in symlinks.items():
    os.makedirs(os.path.dirname(local), exist_ok=True)
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        os.remove(local)
    elif os.path.isdir(local):
        shutil.rmtree(local)
    os.symlink(remote, local)
    print(f'  {local} -> {remote}')

# Check for existing checkpoints (auto-resume)
DRIVE_CKPT = f'{DRIVE_PROJECT}/checkpoints/dynamics'
existing = sorted(glob.glob(f'{DRIVE_CKPT}/step_*.pt'))
if existing:
    print(f'\nFound {len(existing)} checkpoint(s): {[os.path.basename(p) for p in existing]}')
    print('Training will auto-resume from the latest.')
else:
    print('\nNo existing checkpoints. Training will start from scratch.')
print('\nSymlinks ready')

---
## 4. Smoke Test

In [ ]:
!cd /content/minigenie && python scripts/smoke_test.py

---
## 5. Train Dynamics Model

Calls the full training loop from `src/training/train_dynamics.py`.  
Resumes automatically from the latest checkpoint on Drive.

**Monitor:**
- Loss should decrease steadily → expect [0.01, 0.05] range by convergence
- **Step 5K checkpoint:** loss should be clearly decreasing. If flat or NaN → stop & debug.
- **Step 20K checkpoint:** 1-step predictions should look vaguely like CoinRun frames (not noise).
- Sample predictions saved every 5K steps to Drive.

**If session disconnects:** Just re-run this notebook — training resumes from the last checkpoint.

**Override defaults** by passing keyword arguments below.

In [ ]:
from src.training.train_dynamics import train

train(
    data_dir='/content/minigenie/data/coinrun/episodes',
    ckpt_dir='/content/minigenie/checkpoints/dynamics',
    config_path='/content/minigenie/configs/dynamics.yaml',
    # --- Override any config values here ---
    # max_steps=150000,
    # batch_size=16,      # Reduce to 8 on T4, use grad accum x2
    # lr=2e-4,
    resume=True,
    device='cuda',
)

---
## 6. Inspect Sample Predictions

Each sample image shows:
- **Top row:** Ground truth target frames
- **Bottom row:** Model's 1-step predictions

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

samples_dir = f'{DRIVE_PROJECT}/samples_dynamics'
sample_files = sorted(glob.glob(f'{samples_dir}/step_*.png'))

if sample_files:
    print(f'Found {len(sample_files)} sample images')
    for path in sample_files[-3:]:
        img = Image.open(path)
        fig, ax = plt.subplots(1, 1, figsize=(12, 6))
        ax.imshow(img)
        ax.set_title(os.path.basename(path))
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('No sample images yet. Train for at least sample_every steps.')

---
## 7. Generate Multi-Step Rollout

After significant training, generate a multi-step autoregressive rollout to see temporal coherence.

In [ ]:
import torch
import numpy as np
import yaml
import matplotlib.pyplot as plt
from src.models.unet import UNet
from src.training.checkpoint import CheckpointManager
from src.training.train_dynamics import generate_next_frame
from src.data.dataset import WorldModelDataset

# Load config
with open('/content/minigenie/configs/dynamics.yaml') as f:
    cfg = yaml.safe_load(f)
mcfg = cfg['model']
fcfg = cfg['flow']

# Load model
ckpt_mgr = CheckpointManager('/content/minigenie/checkpoints/dynamics')
state = ckpt_mgr.load_latest()

if state is not None:
    model = UNet(
        in_channels=mcfg.get('in_channels', 15),
        out_channels=mcfg.get('out_channels', 3),
        channel_mult=mcfg.get('channel_mult', [64, 128, 256, 512]),
        cond_dim=mcfg.get('cond_dim', 512),
        num_actions=mcfg.get('num_actions', 15),
        num_groups=mcfg.get('num_groups', 32),
        cfg_dropout=0.0,  # No dropout at inference
    ).cuda()
    model.load_state_dict(state['model'])
    model.eval()
    step = state['step']
    print(f'Loaded checkpoint at step {step}')

    # Load a real episode for starting context
    dataset = WorldModelDataset(
        '/content/minigenie/data/coinrun/episodes',
        context_length=mcfg.get('context_frames', 4),
    )
    context, action, target = dataset[0]
    context = context.unsqueeze(0).cuda()  # [1, 12, H, W]

    # Autoregressive rollout
    ROLLOUT_STEPS = 20
    NUM_INFERENCE_STEPS = fcfg.get('num_inference_steps', 15)
    CFG_SCALE = fcfg.get('cfg_scale', 2.0)
    H = mcfg.get('context_frames', 4)

    # Random actions for the rollout
    actions = torch.randint(0, mcfg.get('num_actions', 15), (ROLLOUT_STEPS,)).cuda()

    # Collect context frames (first H frames from the context tensor)
    frames = []
    for i in range(H):
        frames.append(context[0, i*3:(i+1)*3].cpu())  # [3, h, w]

    # Generate frames autoregressively
    print(f'Generating {ROLLOUT_STEPS}-step rollout...')
    with torch.no_grad():
        for t_step in range(ROLLOUT_STEPS):
            # Build context from last H frames
            ctx = torch.cat(frames[-H:], dim=0).unsqueeze(0).cuda()  # [1, H*3, h, w]
            act = actions[t_step:t_step+1]  # [1]
            pred = generate_next_frame(
                model, ctx, act,
                num_steps=NUM_INFERENCE_STEPS,
                cfg_scale=CFG_SCALE,
            )
            frames.append(pred[0].cpu())  # [3, h, w]

    # Display as grid
    total = len(frames)
    cols = min(10, total)
    rows = (total + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2*cols, 2*rows))
    if rows == 1:
        axes = [axes]
    for i, frame in enumerate(frames):
        r, c = i // cols, i % cols
        ax = axes[r][c] if rows > 1 else axes[c]
        img = frame.permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img)
        label = f'ctx {i}' if i < H else f'pred {i-H}'
        ax.set_title(label, fontsize=8)
        ax.axis('off')
    # Hide empty axes
    for i in range(total, rows * cols):
        r, c = i // cols, i % cols
        ax = axes[r][c] if rows > 1 else axes[c]
        ax.axis('off')
    plt.suptitle(f'Autoregressive Rollout (step {step}, {ROLLOUT_STEPS} predicted frames)', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No checkpoint found. Train the model first.')

---
## 8. Compute Single-Step PSNR

In [ ]:
# Re-use model from above (or reload if needed)
if state is not None:
    from torch.utils.data import DataLoader

    dataset = WorldModelDataset(
        '/content/minigenie/data/coinrun/episodes',
        context_length=mcfg.get('context_frames', 4),
    )
    loader = DataLoader(dataset, batch_size=16, shuffle=True)

    NUM_INFERENCE_STEPS = fcfg.get('num_inference_steps', 15)
    CFG_SCALE = fcfg.get('cfg_scale', 2.0)

    psnr_values = []
    with torch.no_grad():
        for i, (ctx, act, tgt) in enumerate(loader):
            if i >= 20:  # 20 x 16 = 320 samples
                break
            ctx, act, tgt = ctx.cuda(), act.cuda(), tgt.cuda()
            pred = generate_next_frame(model, ctx, act, NUM_INFERENCE_STEPS, CFG_SCALE)
            mse = ((tgt - pred) ** 2).mean(dim=(1, 2, 3))
            psnr = -10 * torch.log10(mse + 1e-8)
            psnr_values.extend(psnr.cpu().tolist())
            if (i + 1) % 5 == 0:
                print(f'  Batch {i+1}/20 done...')

    psnr_arr = np.array(psnr_values)
    print(f'\nSingle-step PSNR over {len(psnr_arr)} predictions:')
    print(f'   Mean:   {psnr_arr.mean():.2f} dB')
    print(f'   Median: {np.median(psnr_arr):.2f} dB')
    print(f'   Min:    {psnr_arr.min():.2f} dB')
    print(f'   Max:    {psnr_arr.max():.2f} dB')
    target_psnr = cfg.get('targets', {}).get('single_step_psnr_db', 22.0)
    status = 'PASS' if psnr_arr.mean() >= target_psnr else 'BELOW TARGET'
    print(f'   Target: {target_psnr} dB [{status}]')
else:
    print('No model loaded.')

---
## 9. Post-Session Checklist

After training completes or before Colab disconnects:

1. Checkpoints are on Drive (symlinked)
2. Sample predictions are on Drive (symlinked)
3. Update `logs/TRAINING_LOG.md` with:
   - Date, GPU type, steps completed (from -> to)
   - Final loss value
   - Single-step PSNR
   - Rollout quality (sharp/blurry/collapsed)
   - Any issues or observations
4. Push code changes to GitHub

In [ ]:
# Summary of what's on Drive
print('=== Drive Contents ===')
for subdir in ['checkpoints/dynamics', 'samples_dynamics']:
    full = f'{DRIVE_PROJECT}/{subdir}'
    if os.path.exists(full):
        files = os.listdir(full)
        print(f'  {subdir}/: {len(files)} files')
        for f in sorted(files)[-5:]:
            size_mb = os.path.getsize(os.path.join(full, f)) / 1e6
            print(f'    {f} ({size_mb:.1f} MB)')
    else:
        print(f'  {subdir}/: (not found)')